### Task 1

In [55]:
import pandas as pd
from sklearn.model_selection import train_test_split
 
df = pd.read_csv("cars.csv")

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    398 non-null    object 
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model year    398 non-null    int64  
 7   origin        398 non-null    int64  
 8   car name      398 non-null    object 
dtypes: float64(3), int64(4), object(2)
memory usage: 28.1+ KB


In [57]:
# horsepower is an object but should be a float
# origin is numeric but is categorical

In [58]:
df['horsepower'].describe()

count     398
unique     94
top       150
freq       22
Name: horsepower, dtype: object

### Task 2

In [59]:
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')
df['horsepower'] = df['horsepower'].fillna(df['horsepower'].median())

# the mean and median were close in values, the median was taken to better 
# represent the middle of the dataset values.

In [60]:
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130.0,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,1,ford torino


### Task 3

In [61]:
X = df.drop(columns=['mpg','car name'])
y = df['mpg']

# 'car name' can't be used in the model input because it is not numerical and can't
# be represented numerically. 

### Task 5

In [62]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
ohe = OneHotEncoder(sparse_output=False)
origin_array = ohe.fit_transform(X[['origin']])

origin_cols = ohe.get_feature_names_out(['origin'])
origin_df = pd.DataFrame(origin_array, columns=origin_cols, dtype=int)

df = pd.concat([df, origin_df], axis=1)
df.sample(10)

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name,origin_1,origin_2,origin_3
221,17.5,8,305.0,145.0,3880,12.5,77,1,chevrolet caprice classic,1,0,0
267,27.5,4,134.0,95.0,2560,14.2,78,3,toyota corona,0,0,1
280,21.5,6,231.0,115.0,3245,15.4,79,1,pontiac lemans v6,1,0,0
292,18.5,8,360.0,150.0,3940,13.0,79,1,chrysler lebaron town @ country (sw),1,0,0
127,19.0,6,232.0,100.0,2901,16.0,74,1,amc hornet,1,0,0
39,14.0,8,400.0,175.0,4464,11.5,71,1,pontiac catalina brougham,1,0,0
41,14.0,8,318.0,150.0,4096,13.0,71,1,plymouth fury iii,1,0,0
374,23.0,4,151.0,93.5,3035,20.5,82,1,amc concord dl,1,0,0
176,19.0,6,232.0,90.0,3211,17.0,75,1,amc pacer,1,0,0
13,14.0,8,455.0,225.0,3086,10.0,70,1,buick estate wagon (sw),1,0,0


### Task 4

Min and Max or std deviations are leaked when scaling before splitting as the values for the entire data set are used to calculate scales instead of the isolated test and training sets.

We would be including test data into training which inflates our perception on how well the model is working.

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42)
print(len(y_train))
print(len(y_test))


318
80


### Task 6

In [64]:
X_train.drop(columns=['origin','origin_1'])
X_test.drop(columns=['origin','origin_1'])

KeyError: "['origin_1'] not found in axis"

1. We dropped the `origin` column and the `origin_1` column. It doesn't matter which of the `origin_` columns is dropped because if the values of the others are both 0, you know that it must be 1 in the column that was dropped.
2. This mulitcollinearity is not that the columns are similar or describe similar things, it is defininitionally all the same thing. Numeric features that are multicollinear can affect each other, but these One-Hot columns are not influenced by other features, they're all describing the same feature.

### Task 7

In [ ]:
ss = StandardScaler()

X_train_scaled = ss.fit_transform(X_train[['cylinders','displacement','horsepower','weight','acceleration','model year']])

X_test_scaled = ss.transform(X_test[['cylinders','displacement','horsepower','weight','acceleration','model year']])

print(f'Mean from StandardScaler: {ss.mean_}  ||  \nStd Dev from StandardScalar: {ss.scale_}')

Mean from StandardScaler: [   5.43081761  191.90408805  103.13050314 2969.01572327   15.63993711
   76.10377358]  ||  
Std Dev from StandardScalar: [  1.68229589 102.82175033  37.02696765 839.29496266   2.75892081
   3.59750764]


### Task 8

In [86]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error as MAE

cars_df = pd.read_csv('cars.csv')
nominal_cols = ['origin']
numeric_cols = ['cylinders','displacement','horsepower','weight','acceleration','model year']

preprocessor = ColumnTransformer([
    ('nominal',         OneHotEncoder(sparse_output=False, drop='first'),           nominal_cols),
    ('numeric',         StandardScaler(),                                           numeric_cols)
])

pipe = Pipeline([
    ('preprocessor',    preprocessor),
    ('model',           LinearRegression())
])

pipe.fit(X_train, y_train)
y_preds = pipe.predict(X_test)

r2 = r2_score(y_test, y_preds)
mae = MAE(y_test, y_preds)

print(f'R2 Score: {r2:.2f}')
print(f'MAE: {mae:.2f} MPG')

R2 Score: 0.84
MAE: 2.29 MPG


The pipeline is much more simple. The thing it 'enforces' that we had to be careful about is that the pipeline automatically maintains the train/test split when fitting the scalers.  We had to manually call `fit_transform` for the training set and `transform` for the test set.